# 🧠 ฟังก์ชันกระตุ้น (Activation Functions) และอนุพันธ์

ยินดีต้อนรับสู่สมุดบันทึกคำอธิบายเชิงปฏิบัติสำหรับ **Activation Functions**! ในสมุดบันทึกนี้ เราจะ:
1. กำหนดสูตรทางคณิตศาสตร์สำหรับ Sigmoid, ReLU, Leaky ReLU และ SiLU (Swish)
2. เขียนโค้ดสำหรับฟังก์ชันกระตุ้นและอนุพันธ์จากศูนย์ด้วย NumPy
3. พล็อตและเปรียบเทียบฟังก์ชันกระตุ้นแบบเคียงข้างกัน
4. พล็อตและเปรียบเทียบอนุพันธ์ของฟังก์ชันกระตุ้นเพื่อแสดงภาพปัญหา **เกรเดียนต์หายไป (Vanishing Gradient)** และ **ปัญหาเซลล์ประสาทตาย (Dying ReLU)**
5. อธิบายว่าทำไม YOLO จึงใช้ค่าเริ่มต้นเป็น **SiLU** ในเลเยอร์ convolution ของมัน

มาเริ่มต้นด้วยการนำเข้าไลบรารีที่จำเป็นกันก่อน

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Set seed for reproducibility
np.random.seed(42)

## 1. การสร้างฟังก์ชันกระตุ้นและอนุพันธ์ขึ้นมาใช้เองด้วย NumPy

มาเขียนโค้ดสำหรับฟังก์ชันและอนุพันธ์ของมันกัน:
- **Sigmoid:** $\sigma(x) = \frac{1}{1 + e^{-x}}$
- **ReLU:** $\max(0, x)$
- **Leaky ReLU:** $\max(0.1x, x)$
- **SiLU:** $x \cdot \sigma(x)$

In [ ]:
def sigmoid(x):
    return 1.0 / (1.0 + np.exp(-x))

def sigmoid_derivative(x):
    s = sigmoid(x)
    return s * (1.0 - s)

def relu(x):
    return np.maximum(0.0, x)

def relu_derivative(x):
    return np.where(x > 0.0, 1.0, 0.0)

def leaky_relu(x, alpha=0.1):
    return np.maximum(alpha * x, x)

def leaky_relu_derivative(x, alpha=0.1):
    return np.where(x > 0.0, 1.0, alpha)

def silu(x):
    return x * sigmoid(x)

def silu_derivative(x):
    sig = sigmoid(x)
    return sig * (1.0 + x * (1.0 - sig))

## 2. การแสดงภาพเปรียบเทียบ ฟังก์ชันกระตุ้น vs. อนุพันธ์

มาพล็อตฟังก์ชันกระตุ้นทางด้านซ้าย และอนุพันธ์ของฟังก์ชันกระตุ้นทางด้านขวา เพื่อวิเคราะห์การไหลของเกรเดียนต์ในขั้นตอน Backpropagation

In [ ]:
x = np.linspace(-6, 6, 300)

plt.figure(figsize=(16, 7))

# Plot Activation Functions
plt.subplot(1, 2, 1)
plt.plot(x, sigmoid(x), color='purple', linewidth=2.5, label='Sigmoid')
plt.plot(x, relu(x), color='red', linewidth=2.5, label='ReLU')
plt.plot(x, leaky_relu(x), color='green', linewidth=2.5, label='Leaky ReLU (α=0.1)')
plt.plot(x, silu(x), color='teal', linewidth=3, label='SiLU (Swish)')
plt.ylim(-2, 5)
plt.xlabel('Input (x)')
plt.ylabel('Activation Output')
plt.title('Activation Functions')
plt.grid(True, linestyle='--', alpha=0.5)
plt.legend()

# Plot Derivatives
plt.subplot(1, 2, 2)
plt.plot(x, sigmoid_derivative(x), color='purple', linewidth=2.5, label='Sigmoid Derivative')
plt.plot(x, relu_derivative(x), color='red', linewidth=2.5, label='ReLU Derivative')
plt.plot(x, leaky_relu_derivative(x), color='green', linewidth=2.5, label='Leaky ReLU Derivative')
plt.plot(x, silu_derivative(x), color='teal', linewidth=3, label='SiLU Derivative')
plt.ylim(-0.2, 1.2)
plt.xlabel('Input (x)')
plt.ylabel('Gradient / Slope')
plt.title('Derivatives (Gradient Flow)')
plt.grid(True, linestyle='--', alpha=0.5)
plt.legend()

plt.tight_layout()
plt.show()

## 3. ข้อสังเกตและการวินิจฉัยที่สำคัญ

*   **Sigmoid (ปัญหาเกรเดียนต์หายไป - Vanishing Gradient):** เมื่อ $|x| > 4$ ค่าอนุพันธ์จะลดลงจนเกือบเป็นศูนย์ ในโครงข่ายประสาทที่ลึกมากๆ การคูณค่าที่น้อยมากเหล่านี้ในแต่ละชั้นต่อๆ กันจะทำให้เกรเดียนต์ของชั้นแรกๆ หายไปอย่างสิ้นเชิง ส่งผลให้การเรียนรู้หยุดลง
*   **ReLU (ปัญหาเซลล์ประสาทตาย - Dying ReLU):** สำหรับค่าอินพุตที่เป็นลบทั้งหมด ($x < 0$) ค่าอนุพันธ์จะเป็น 0 พอดี หากเซลล์ประสาทถูกอัปเดตจนทำให้ได้รับค่าลบเสมอ เซลล์นั้นจะไม่ได้รับการอัปเดตอีกเลย ("ตาย")
*   **Leaky ReLU (การป้องกันปัญหาเซลล์ประสาทตาย):** รักษาความชันขนาดเล็กไว้ ($0.1$ หรือ $0.01$) สำหรับค่าอินพุตที่เป็นลบ ช่วยให้เกรเดียนต์สามารถไหลกลับได้แม้กระทั่งในเซลล์ประสาทที่ไม่ทำงาน
*   **SiLU (การไหลของเกรเดียนต์แบบราบรื่น - Smooth Gradient):** ค่าเริ่มต้นใน YOLO มีเส้นโค้งที่ราบรื่นและมีอนุพันธ์ที่ต่อเนื่องกัน (ไม่มีเหลี่ยมมุมที่แหลมคม ณ จุด $x=0$) รวมถึงมีหุบเขาลบขนาดเล็ก (small negative valley) สิ่งนี้ช่วยป้องกันปัญหาเซลล์ประสาทตายในขณะเดียวกันก็ช่วยรักษาสภาพความเสถียรในการอัปเดตของแต่ละ batch